In [1]:
import os
import numpy as np
import geopandas as gpd
import rasterio
from rasterio import features
import flopy
from flopy.utils import Raster
from flopy.utils.gridintersect import GridIntersect
from shapely.geometry import Polygon


In [2]:
cwd = os.getcwd()
# Set up directories and paths
data_dir = os.path.join(cwd, '..', 'data')
processed_gis_dir = os.path.join(cwd, '..', 'gisfiles', 'processed_gis')
modflow_dir = os.path.join(cwd, "..", "model")

# Create modflow directory if it doesn't exist
os.makedirs(modflow_dir, exist_ok=True)

# Load topography grid
topo_asc = os.path.join(processed_gis_dir, 'top.asc')
topo_grid = np.loadtxt(topo_asc, skiprows=6)

# Get grid dimensions
nrows, ncols = topo_grid.shape
print(f"Grid dimensions: {nrows} rows x {ncols} columns")

# Read the ASCII grid header to get cell size and bounds
with open(topo_asc, 'r') as f:
    lines = [f.readline() for _ in range(6)]
    ncols_h = int(lines[0].split()[1])
    nrows_h = int(lines[1].split()[1])
    xllcorner = float(lines[2].split()[1])
    yllcorner = float(lines[3].split()[1])
    cellsize = float(lines[4].split()[1])

delr = np.full(ncols, cellsize)  # Column widths
delc = np.full(nrows, cellsize)  # Row widths

# Create 3-layer model
nlay = 3
nper = 1  # Number of stress periods

# Define layer bottom elevations (roughly 100m intervals below surface)
top_elev = topo_grid  # Top of model = topography
bot1 = top_elev - 100   # Bottom of layer 1
bot2 = bot1 - 30      # Bottom of layer 2
bot3 = bot2 - 100      # Bottom of layer 3

botm = np.array([bot1, bot2, bot3])

# Load idomain file (ibound.dat) and apply to all 3 layers
ibound_2d = np.loadtxt(os.path.join(processed_gis_dir, 'ibound.dat'), dtype=int)
idomain = np.zeros((nlay, nrows, ncols), dtype=int)
for k in range(nlay):
    idomain[k] = ibound_2d

# Create MODFLOW-6 simulation
modelname = 'minet01'
sim = flopy.mf6.MFSimulation(sim_name=modelname, sim_ws=modflow_dir, exe_name='mf6.exe')

# Create time discretization
tdis = flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=[(1.0, 1, 1.0)], time_units='days')

# Create groundwater flow model
gwf = flopy.mf6.ModflowGwf(sim, modelname=modelname, save_flows=True)

# Create discretization package
dis = flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=nlay,
    nrow=nrows,
    ncol=ncols,
    delr=delr,
    delc=delc,
    top=top_elev,
    botm=botm,
    idomain=idomain,
)

start_head = np.full((nlay, nrows, ncols), 1.0)
start_head[0] = topo_grid  # Start heads in layer 1 = topography
start_head[1] = bot1
  # Start heads in layer 2 = bottom of layer 1
start_head[2] = bot2
  # Start heads in layer 3 = bottom of layer 2
# Create initial conditions package
ic = flopy.mf6.ModflowGwfic(gwf, strt=start_head)

# Load conductivity data (example: uniform or from file)
# Adjust based on available ASCII grids for conductivity
hk = np.full((nlay, nrows, ncols), 1.0)  # Horizontal conductivity (m/day)
vk = np.full((nlay, nrows, ncols), 0.1)  # Vertical conductivity (m/day)

# Layered properties example
hk[0] = np.loadtxt(os.path.join(processed_gis_dir, 'hk1.dat'))  # Layer 1 - higher K from file
hk[1] = np.loadtxt(os.path.join(processed_gis_dir, 'hk2.dat'))  # Layer 2 - lower K from file
hk[2] = np.loadtxt(os.path.join(processed_gis_dir, 'hk3.dat'))  # Layer 3 - medium K from file

# Create node property flow package
npf = flopy.mf6.ModflowGwfnpf(gwf, k=hk, k33=vk, save_flows=True)

# Load river locations from grid
riv_grid = np.loadtxt(os.path.join(processed_gis_dir, 'riv_grid.dat'))

# Create river package based on riv field
riv_cells = []
if os.path.exists(os.path.join(processed_gis_dir, 'riv_grid.dat')):
    for i in range(nrows):
        for j in range(ncols):
            if riv_grid[i, j] == 1:
                # River in layer 1 (top layer)
                riv_elev = top_elev[i, j] - 5  # Riverbed 5m below surface
                riv_cond = 10.0  # Conductance
                riv_cells.append([(0, i, j), riv_elev, riv_cond, riv_elev])

if riv_cells:
    riv = flopy.mf6.ModflowGwfriv(gwf, stress_period_data={0: riv_cells})
    print(f"Created river package with {len(riv_cells)} cells")

# Create well package with specified fluxes (negative for extraction)
well_entries = [
    (1, 25, 17, -19),
    (3, 54, 44, -13),
    (3, 78, 100, -15),
    (3, 35, 44, -13),
    (1, 54, 44, -7),
]

stress_period_data = [
    ((layer - 1, row - 1, col - 1), flux)
    for layer, row, col, flux in well_entries
]

wel = flopy.mf6.ModflowGwfwel(
    gwf,
    stress_period_data={0: stress_period_data},
)

#create a recharge package that applies recharge to the top layer at a rate of 0.001 m/day
recharge_rate = 0.001  # m/day
# rch = flopy.mf6.ModflowGwfrch(gwf, recharge=recharge_rate)
rch = flopy.mf6.ModflowGwfrcha(
    gwf,
    pname='rch',
    recharge=recharge_rate,
    filename='minet01.rcha'
)

# Create output control package
oc = flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord=f"{modelname}.hds",
    budget_filerecord=f"{modelname}.cbc",
    saverecord=[('HEAD', 'ALL'), ('BUDGET', 'ALL')],
)


# Create IMS solver
ims = flopy.mf6.ModflowIms(sim, complexity='SIMPLE')

# Write model files
sim.write_simulation()
print(f"\nMODFLOW-6 model created in: {modflow_dir}")
print(f"Model name: {modelname}")

Grid dimensions: 105 rows x 131 columns
Created river package with 1824 cells
writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_-1...
  writing model minet01...
    writing model name file...
    writing package dis...
    writing package ic...
    writing package npf...
    writing package riv_0...
INFORMATION: maxbound in ('', 'riv', 'dimensions') changed to 1824 based on size of stress_period_data
    writing package wel_0...
INFORMATION: maxbound in ('', 'wel', 'dimensions') changed to 5 based on size of stress_period_data
    writing package rch...
    writing package oc...

MODFLOW-6 model created in: c:\Users\mouss\modeling\gitrepo\mine-t01\scripts\..\model
Model name: minet01


In [3]:
# run the simulation using the "runmf6.bat" file created in the modflow directory
# 
run_command = os.path.join(modflow_dir, "runmf6.bat")
if os.path.exists(run_command):
    print(f"\nRunning MODFLOW-6 simulation using: {run_command}")
    os.system(run_command)
else:
    print(f"\nRun command not found: {run_command}")
    print("Please run the simulation manually using the generated batch file.")



Running MODFLOW-6 simulation using: c:\Users\mouss\modeling\gitrepo\mine-t01\scripts\..\model\runmf6.bat
